In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import norm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss
)

In [ ]:
def prepare_datasets(df, target, df_test, feature_cols):
    """
    Prépare les jeux de données pour l'entraînement,
    l'évaluation et la prédiction.

    Retourne un dictionnaire contenant :
    - X_train, y_train : données d'entraînement
    - X_test, y_test : données de test complètes
    - X_test_all, y_test_all : données de test avant filtrage
    - missing_df : lignes de df dont la cible est manquante
    - X_missing : variables explicatives des lignes à prédire
    - train_df : lignes de df dont la cible est connue
    - test_known : lignes de test dont la cible est connue
    """

    # Vérifier les colonnes nécessaires
    required_train_cols = set(feature_cols + [target])
    missing_train_cols = required_train_cols - set(df.columns)

    if missing_train_cols:
        raise ValueError(
            f"Colonnes absentes de df : {sorted(missing_train_cols)}"
        )

    required_test_cols = required_train_cols
    missing_test_cols = required_test_cols - set(df_test.columns)

    if missing_test_cols:
        raise ValueError(
            f"Colonnes absentes de df_test : {sorted(missing_test_cols)}"
        )

    # Aligner les colonnes du test sur celles de l'entraînement
    df_test = df_test[feature_cols + [target]].copy()

    # Séparer les lignes selon la présence de la cible
    train_df = df[df[target].notna()].copy()
    missing_df = df[df[target].isna()].copy()

    # Construire X et y pour l'entraînement
    X_train_all = train_df[feature_cols]
    y_train_all = train_df[target]

    # Conserver uniquement les lignes d'entraînement complètes
    train_mask = X_train_all.notna().all(axis=1)

    X_train = X_train_all.loc[train_mask].copy()
    y_train = y_train_all.loc[train_mask].copy()

    # Préparer les données de test dont la cible est connue
    test_known = df_test[df_test[target].notna()].copy()

    X_test_all = test_known[feature_cols]
    y_test_all = test_known[target]

    # Conserver uniquement les lignes de test complètes
    test_mask = X_test_all.notna().all(axis=1)

    X_test = X_test_all.loc[test_mask].copy()
    y_test = y_test_all.loc[test_mask].copy()

    # Préparer les lignes dont la cible doit être prédite
    X_missing_all = missing_df[feature_cols]

    # Une prédiction n'est possible ici que si les variables
    # explicatives sont toutes renseignées
    missing_mask = X_missing_all.notna().all(axis=1)
    X_missing = X_missing_all.loc[missing_mask].copy()

    # Retourner tous les jeux de données utiles
    return {
        "train_df": train_df,
        "missing_df": missing_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "X_test_all": X_test_all,
        "y_test_all": y_test_all,
        "test_known": test_known,
        "X_missing": X_missing,
        "missing_mask": missing_mask,
        "train_mask": train_mask,
        "test_mask": test_mask,
    }

In [ ]:
def build_model(X_train):
    # Identifier les types de colonnes
    cat_cols = X_train.select_dtypes(
        include=["object", "category", "string", "bool"]
    ).columns.tolist()

    num_cols = X_train.select_dtypes(
        include="number"
    ).columns.tolist()

    # Prétraitement des variables catégorielles
    cat_pipeline = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Prétraitement des variables numériques
    num_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    # Combiner les prétraitements
    transformers = []

    if cat_cols:
        transformers.append(("cat", cat_pipeline, cat_cols))

    if num_cols:
        transformers.append(("num", num_pipeline, num_cols))

    preprocessor = ColumnTransformer(
        transformers=transformers
    )

    # Construire le pipeline complet
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ])

    return model


In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    Évalue un modèle entraîné sur le jeu de test.
    Affiche son taux de réussite et renvoie les métriques.
    """

    # Vérifier que le jeu de test contient des données
    if X_test.empty:
        print("Évaluation impossible : le jeu de test est vide.")
        return None

    if len(X_test) != len(y_test):
        raise ValueError(
            "X_test et y_test doivent contenir le même nombre de lignes."
        )

    # Prédictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # Métriques
    accuracy = accuracy_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    classes = model.named_steps["classifier"].classes_

    metrics = {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "classification_report": classification_report(
            y_test,
            y_pred,
            zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            y_test,
            y_pred,
            labels=classes
        )
    }

    # Calculer la log-loss si les classes du test sont connues du modèle
    if set(y_test.unique()).issubset(set(classes)):
        metrics["log_loss"] = log_loss(
            y_test,
            y_proba,
            labels=classes
        )

    # Afficher les résultats
    print(f"Taux de réussite : {accuracy:.2%}")
    print(f"Balanced accuracy : {balanced_accuracy:.2%}")

    print("\nRapport de classification :")
    print(metrics["classification_report"])

    print("Matrice de confusion :")
    print(metrics["confusion_matrix"])


In [ ]:
def fill_missing_values(df, target, df_test):

    # Vérifier que les colonnes sont compatibles
    if target not in df.columns or target not in df_test.columns:
        raise ValueError(f"La colonne {target!r} doit exister dans les deux DataFrames.")

    feature_cols = [col for col in df.columns if col != target]

    if set(feature_cols) != set(
        col for col in df_test.columns if col != target
    ):
        raise ValueError("Les variables explicatives de df et df_test doivent être identiques.")

    datasets = prepare_datasets(
        df=df,
        target=target,
        df_test=df_test,
        feature_cols=feature_cols
    )

    X_train = datasets["X_train"]
    y_train = datasets["y_train"]

    X_test = datasets["X_test"]
    y_test = datasets["y_test"]

    X_missing = datasets["X_missing"]
    missing_df = datasets["missing_df"]

    # 1. Construire le modèle
    model = build_model(X_train)

    # 2. Entraîner le modèle
    model.fit(X_train, y_train)

    evaluate_model(model, X_test, y_test)

    predictions = model.predict(X_missing)

    return {
        "model": model,
        "probabilities": probabilities,
        "predictions": prediction_df
    }
